### Dataset and Task Metadata

In [12]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="leukemia_allaml",
    dataset_year="1999",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://www.openml.org/d/1104",
    download_description="""
We download the dataset from OpenML to a predefined folder.

mkdir -p local-data-warehouse/leukemia_allaml/ && wget -P local-data-warehouse/leukemia_allaml/ https://www.openml.org/data/download/53987/leukemia.arff
""",
    # References
    academic_reference_bibtex="""@article{Golub1999MolecularCC,
  title={Molecular classification of cancer: class discovery and class prediction by gene expression monitoring},
  author={Todd R. Golub and Donna K. Slonim and Pablo Tamayo and Christine Huard and Michelle Gaasenbeek and Jill P. Mesirov and Hilary Coller and Mignon L. Loh and James R. Downing and Michael A. Caligiuri and Clara D. Bloomfield and Eric S. Lander},
  journal={Science},
  year={1999},
  volume={286},
  number={5439},
  pages={531-537},
  doi={10.1126/science.286.5439.531},
  url={https://doi.org/10.1126/science.286.5439.531}
}
""",
    academic_reference_bibtex_key="Golub1999MolecularCC",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
        - We use OpenML repository version as the raw data is not available. This dataset contains 7,129 features while the original reference mentions only 6,817 genes. This discrepancy is likely due to some gene filtering based on gene expression level that has not been disclosed in the paper. Hence, we treat OpenML version as the raw, untreated data.
        - We remove 57 columns corresponding to Affymetrix control probes that are not relevant for the prediction task.
        - We shuffle merged samples as the first 38 instances come from the training set.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LeukemiaType",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="LeukemiaType",
)

## Preprocessing

In [13]:
import pandas as pd
import arff

with open(dataset_mold.path/"leukemia.arff") as f:
    data = arff.load(f)
df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

# filter out affymetrix control probes
df.drop([col for col in df.columns if col.startswith('AFFX-')], axis=1, inplace=True)

target_feature = "LeukemiaType"
df = df.rename(columns={"CLASS": target_feature})
df[target_feature] = df[target_feature].astype("category")

df = df.sample(frac=1, random_state=453).reset_index(drop=True)

## Data Checks

In [14]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 72
Columns: 7072
Use sampling: False (sample size: 72)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['HG384-HT384_at', 'D50310_at', 'X04347_s_at', 'U21689_at', 'U72649_at', 'X71874_cds1_at', 'U17566_at', 'Y08765_s_at', 'L16842_at', 'U94832_at']
Rows remaining as candidates after top-10 filter: 0 (of 72)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [15]:
# Sample Rows
df_head

hum_alu_at  A28102_at  AB000114_at  AB000115_at  AB000220_at  AB000409_at  \
0     18343.0      420.0        154.0       1559.0         32.0         70.0   
1     11195.0      260.0         28.0        128.0         46.0          0.0   
2     25673.0      485.0         87.0        178.0        162.0        -57.0   
3     43084.0      267.0         12.0         84.0        -24.0       -184.0   
4     23505.0       85.0         49.0        197.0         13.0         30.0   

   AB000449_at  AB000450_at  AB000460_at  AB000462_at  AB000464_at  \
0        208.0         80.0       1404.0        134.0        481.0   
1        833.0        390.0       2222.0        246.0        592.0   
2        197.0        115.0       2325.0        161.0       1125.0   
3         89.0        117.0       1333.0         57.0        347.0   
4        165.0        166.0       1288.0        136.0        839.0   

   AB000466_at  AB000467_at  AB000468_at  AB000584_at  AB000895_at  \
0       -418.0       -351.0       -112.0        106.0       -144.0   
1      -1046.0      -1123.0       1766.0       -420.0       -637.0   
2      -1693.0      -1871.0       -345.0       -190.0      -1157.0   
3       -893.0       -464.0        314.0       -504.0          4.0   
4       -397.0       -619.0        712.0       -288.0        172.0   

   AB000896_at  AB000897_at  AB000905_at  AB001106_at  AB001325_at  \
0        -83.0        -92.0        138.0        169.0        847.0   
1        -59.0        -57.0        138.0        863.0        872.0   
2       -104.0       -181.0        262.0        309.0       1561.0   
3        -69.0        -52.0          7.0        204.0        654.0   
4         55.0        -86.0        118.0        182.0        605.0   

   AB002314_at  AB002315_at  AB002318_at  AB002365_at  AB002366_at  \
0         50.0        145.0        606.0        -64.0        -39.0   
1       -139.0        285.0       1652.0         -8.0         38.0   
2        137.0         20.0       1891.0        208.0         46.0   
3         92.0        159.0        828.0        -24.0        -39.0   
4         46.0        179.0        434.0        -30.0        -17.0   

   AB002380_at  AB002382_at  AB002409_at  AB002559_at  AB003102_at  \
0          4.0        430.0        121.0        667.0        388.0   
1         35.0        583.0       -103.0        708.0       1263.0   
2        106.0        414.0       -172.0       1187.0        627.0   
3        -19.0        474.0         72.0        958.0        401.0   
4         73.0        365.0          0.0        916.0        583.0   

   AB003103_at  AB003177_at  AB003698_at  AB004884_at  AB006190_at  \
0         26.0        365.0         61.0        791.0        548.0   
1        445.0       1019.0        705.0       1655.0        852.0   
2        104.0        859.0        150.0       1297.0        921.0   
3         79.0        361.0        154.0        396.0        474.0   
4         52.0        342.0        178.0        611.0        643.0   

   AC000061_cds2_at  AC000061_cds3_at  AC000062_at  AC000064_cds1_at  \
0             225.0            -107.0        -10.0             252.0   
1             149.0            -149.0        -16.0             402.0   
2             882.0            -190.0         42.0             874.0   
3             366.0             -52.0         49.0             272.0   
4             239.0            -143.0         13.0             289.0   

   AC000064_cds2_at  AC000066_at  AC000099_at  AC000115_cds1_at  \
0              93.0        -21.0         33.0            -167.0   
1             218.0        -39.0        186.0            -175.0   
2             175.0        -23.0        336.0              81.0   
3             204.0        -99.0        124.0               4.0   
4              98.0        -76.0         97.0            -121.0   

   AC002045_xpt1_at  AC002073_cds1_at  AC002077_at  AC002086_at  \
0             167.0             253.0       -157.0         59.0   
1             257.0       

In [16]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,LeukemiaType,category,0.0,0.0,2.0,"ALL, AML"
1,hum_alu_at,float64,0.0,0.0,72.0,"18343.0, 11195.0, 12801.0, 14030.0, 25147.0, 24611.0, 34207.0, 11631.0, 16738.0, 29906.0"
2,A28102_at,float64,0.0,0.0,66.0,"260.0, 472.0, 430.0, 118.0, 249.0, 88.0, 420.0, 402.0, 270.0, 20.0"
3,AB000114_at,float64,0.0,0.0,58.0,"36.0, 28.0, -38.0, -23.0, 68.0, -27.0, 1.0, 45.0, 39.0, 0.0"
4,AB000115_at,float64,0.0,0.0,63.0,"197.0, 131.0, 191.0, 74.0, 183.0, 174.0, 99.0, 215.0, 148.0, 113.0"
5,AB000220_at,float64,0.0,0.0,51.0,"39.0, 32.0, 68.0, 51.0, -7.0, 29.0, 35.0, 46.0, 41.0, 43.0"
6,AB000409_at,float64,0.0,0.0,68.0,"125.0, -42.0, 87.0, -103.0, -520.0, -112.0, 146.0, 46.0, 27.0, 70.0"
7,AB000449_at,float64,0.0,0.0,68.0,"285.0, 70.0, 115.0, 208.0, 201.0, 232.0, 498.0, 125.0, 37.0, 169.0"
8,AB000450_at,float64,0.0,0.0,63.0,"259.0, 199.0, 115.0, 54.0, 113.0, 186.0, 173.0, 207.0, 556.0, 179.0"
9,AB000460_at,float64,0.0,0.0,71.0,"2117.0, 1404.0, 1910.0, 3172.0, 2911.0, 1505.0, 1354.0, 1770.0, 1732.0, 1249.0"


In [17]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
hum_alu_at,72.0,25209.180556,10882.622106,6479.0,58203.0
A28102_at,72.0,278.013889,189.318979,20.0,1106.0
AB000114_at,72.0,30.805556,66.629655,-212.0,249.0
AB000115_at,72.0,296.250000,373.146414,65.0,2194.0
AB000220_at,72.0,44.444444,47.778506,-61.0,202.0
AB000409_at,72.0,24.722222,203.801286,-520.0,845.0
AB000449_at,72.0,214.208333,182.164756,-41.0,971.0
AB000450_at,72.0,179.222222,114.225404,2.0,556.0
AB000460_at,72.0,1824.083333,562.957826,714.0,3570.0
AB000462_at,72.0,143.666667,144.610502,-112.0,566.0


In [18]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column       rank                    
LeukemiaType 1      ALL     47  65.28
             2      AML     25  34.72

In [19]:
# Target Distribution
target_df

,count,pct
LeukemiaType,,
ALL,47,65.28
AML,25,34.72


## Task Curation

In [20]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [21]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [22]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to leukemia_allaml/019d9d0f-09f2-7aca-8924-fd2db0f15985
019d9d0f-09f2-7aca-8924-fd2db0f15985
113ad398e937622ebdd161b3dea296ed37c5e7b6b91d560bb4ab53dfe69cdb7a
